# Module 6 Homework

For this homework we will be using the Yellow 2025-11 data from the official website:

In [4]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-03-08 14:36:21--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 2600:9000:2684:7000:b:20a5:b140:21, 2600:9000:2684:cc00:b:20a5:b140:21, 2600:9000:2684:f200:b:20a5:b140:21, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|2600:9000:2684:7000:b:20a5:b140:21|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-11.parquet’

yellow_tripdata_202 100%[===================>]  67.84M  6.99MB/s    in 8.5s    

2026-03-08 14:36:30 (8.03 MB/s) - ‘yellow_tripdata_2025-11.parquet’ saved [71134255/71134255]



Need to import .env variable to find Java runtime through Jupyter venv:

In [1]:
from dotenv import load_dotenv
load_dotenv()  # loads JAVA_HOME from .env into os.environ before SparkSession starts


True

## Question 1: Install Spark and PySpark

- Install Spark
- Run PySpark
- Create a local spark session
- Execute spark.version.

What's the output?

In [2]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder.master("local[*]").appName("ny_taxi").getOrCreate()

spark.version

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/08 15:46:57 WARN Utils: Your hostname, Lorenzos-Mac-mini.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.185 instead (on interface en1)
26/03/08 15:46:57 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/08 15:46:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


'4.1.1'

## Question 2: Yellow November 2025

Read the November 2025 Yellow into a Spark Dataframe.

Repartition the Dataframe to 4 partitions and save it to parquet.

What is the average size of the Parquet (ending with .parquet extension) Files that were created (in MB)?

In [5]:
df = spark.read.option("header", "true").parquet("yellow_tripdata_2025-11.parquet")

df.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

In [6]:
df = df.repartition(4)

df.write.parquet("yellow_tripdata_2025-11_repartitioned.parquet")

In [7]:
!ls -lh yellow_tripdata_2025-11_repartitioned.parquet

total 205312
-rw-r--r--@ 1 lorenzoederone  staff     0B Mar  8 14:42 _SUCCESS
-rw-r--r--@ 1 lorenzoederone  staff    24M Mar  8 14:42 part-00000-83620fe0-e4cd-4626-98aa-af41b33af599-c000.snappy.parquet
-rw-r--r--@ 1 lorenzoederone  staff    24M Mar  8 14:42 part-00001-83620fe0-e4cd-4626-98aa-af41b33af599-c000.snappy.parquet
-rw-r--r--@ 1 lorenzoederone  staff    24M Mar  8 14:42 part-00002-83620fe0-e4cd-4626-98aa-af41b33af599-c000.snappy.parquet
-rw-r--r--@ 1 lorenzoederone  staff    24M Mar  8 14:42 part-00003-83620fe0-e4cd-4626-98aa-af41b33af599-c000.snappy.parquet


## Question 3: Count records

How many taxi trips were there on the 15th of November?

Consider only trips that started on the 15th of November.

In [4]:
df = spark.read.parquet("yellow_tripdata_2025-11_repartitioned.parquet/")

df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [10]:
df.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       2| 2025-11-07 15:04:17|  2025-11-07 15:39:15|              1|          7.3|         1|                 N|         262|    

In [5]:
from pyspark.sql.functions import to_date, col

count_2025_11_15 = df.filter(to_date(col("tpep_pickup_datetime")) == "2025-11-15").count()
print(count_2025_11_15)

162604


## Question 4: Longest trip

What is the length of the longest trip in the dataset in hours?

In [6]:
from pyspark.sql.functions import unix_timestamp, max

df = df.withColumn("trip_hours",(unix_timestamp(col("tpep_dropoff_datetime")) - unix_timestamp(col("tpep_pickup_datetime"))) / 3600.0)

# df.select("tpep_pickup_datetime","tpep_dropoff_datetime","trip_hours").show(5)

max_hours = df.select(max("trip_hours")).first()[0]
print(max_hours)

90.64666666666666


## Question 5: User Interface

Spark's User Interface which shows the application's dashboard runs on which local port?

http://localhost:4040

## Question 6: Least frequent pickup location zone

Load the zone lookup data into a temp view in Spark:

In [16]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-08 15:28:33--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 2600:9000:2684:6a00:b:20a5:b140:21, 2600:9000:2684:1600:b:20a5:b140:21, 2600:9000:2684:7000:b:20a5:b140:21, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|2600:9000:2684:6a00:b:20a5:b140:21|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2026-03-08 15:28:33 (203 MB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



Using the zone lookup data and the Yellow November 2025 data, what is the name of the LEAST frequent pickup location Zone?

In [7]:
df_zones = spark.read.option("header", "true").csv("taxi_zone_lookup.csv")
df_zones.createOrReplaceTempView("zones")
df_zones = spark.table("zones")

In [8]:
df_join = df.join(df_zones, df.PULocationID == col("LocationID"), "left")
df_join = df_join.withColumnRenamed("Zone", "pickup_zone")

In [ ]:
df_join.groupBy("pickup_zone").count().orderBy("count").show()

+--------------------+-----+
|         pickup_zone|count|
+--------------------+-----+
|Governor's Island...|    1|
|       Arden Heights|    1|
|Eltingville/Annad...|    1|
|       Port Richmond|    3|
|   Rossville/Woodrow|    4|
| Green-Wood Cemetery|    4|
|         Great Kills|    4|
|       Rikers Island|    4|
|         Jamaica Bay|    5|
|         Westerleigh|   12|
|New Dorp/Midland ...|   14|
|       West Brighton|   14|
|             Oakwood|   14|
|        Crotona Park|   14|
|       Willets Point|   15|
|Breezy Point/Fort...|   16|
|Saint George/New ...|   17|
|       Broad Channel|   18|
|     Mariners Harbor|   21|
|Heartland Village...|   22|
+--------------------+-----+
only showing top 20 rows
